# U3-01 · Regresión: modelos para los alquileres de bicicletas

*Especialista en Inteligencia Artificial (IFCD107) · Unidad 3*

Ejecuta cada celda con **Mayús + Intro**. Todo funciona en el navegador; la primera vez tarda unos segundos en cargar las bibliotecas.

> Datos: Fanaee-T, H. y Gama, J. (2014). Event labeling combining ensemble detectors and background knowledge. *Progress in Artificial Intelligence*, 2, 113–127. Bike Sharing, UCI, CC BY 4.0, https://doi.org/10.24432/C5W894

## 3.2 · Regresión: cuántas bicicletas se alquilarán

`h1_lineal.py`

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from bicis import X, y, h, ent, pru

# referencia: la media de su hora y tipo de día en el entrenamiento
clave = h.hr.astype(str) + "-" + h.workingday.astype(str)
media = y[ent].groupby(clave[ent]).mean()
ref = clave[pru].map(media)
cols = ["temp_c", "humedad", "viento_kmh", "tiempo", "hora_sen",
        "hora_cos", "laborable", "año"]
lin = LinearRegression().fit(X.loc[ent, cols], y[ent])
p = lin.predict(X.loc[pru, cols])
for nombre, pred in [("media por hora y día", ref),
                     ("regresión lineal", p)]:
    print(f"{nombre:<21} EAM {mean_absolute_error(y[pru], pred):6.1f}"
          f"  R² {r2_score(y[pru], pred):.3f}")
print("coeficientes:")
for c, b in zip(cols, lin.coef_):
    print(f"  {c:<11}{b:7.1f}")
print("previsiones negativas:", int((p < 0).sum()), "de", len(p))

`h2_logaritmo.py`

In [ ]:
import numpy as np
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_absolute_error, r2_score
from bicis import X, y, ent, pru

prep = make_column_transformer(
    (OneHotEncoder(), ["hora", "dia_semana", "tiempo", "mes"]),
    (PolynomialFeatures(2), ["temp_c", "humedad", "viento_kmh"]),
    remainder="passthrough")
cols = ["hora", "dia_semana", "tiempo", "mes", "temp_c", "humedad",
        "viento_kmh", "laborable", "año"]
for nombre, m in [
        ("ridge", make_pipeline(prep, Ridge(1.0))),
        ("ridge sobre log(1 + y)", TransformedTargetRegressor(
            make_pipeline(prep, Ridge(1.0)),
            func=np.log1p, inverse_func=np.expm1))]:
    p = m.fit(X.loc[ent, cols], y[ent]).predict(X.loc[pru, cols])
    print(f"{nombre:<23} EAM {mean_absolute_error(y[pru], p):6.1f}"
          f"  R² {r2_score(y[pru], p):.3f}"
          f"  negativas {(p < 0).sum()}")

`h7_regularizar.py`

In [ ]:
import numpy as np
from sklearn.compose import make_column_transformer
from sklearn.linear_model import Ridge, Lasso
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import (OneHotEncoder, PolynomialFeatures,
                                   StandardScaler)
from sklearn.metrics import mean_absolute_error
from bicis import X, y, ent, pru

# interacción hora x laborable x tiempo: muchas columnas
X2 = X.assign(hl=X.hora.astype(str) + "-" + X.laborable.astype(str))
prep = make_column_transformer(
    (OneHotEncoder(handle_unknown="ignore"), ["hl", "mes", "tiempo"]),
    (make_pipeline(StandardScaler(), PolynomialFeatures(3)),
     ["temp_c", "humedad", "viento_kmh"]), remainder="drop")
Ze = prep.fit_transform(X2[ent]); Zp = prep.transform(X2[pru])
print("columnas:", Ze.shape[1])
ly = np.log1p(y)
for nombre, m in [("ridge", Ridge), ("lasso", Lasso)]:
    for a in ([0.01, 1, 100] if nombre == "ridge"
              else [0.0005, 0.005, 0.05]):
        f = m(alpha=a, max_iter=5000).fit(Ze, ly[ent])
        e = mean_absolute_error(y[ent], np.expm1(f.predict(Ze)))
        p = mean_absolute_error(y[pru], np.expm1(f.predict(Zp)))
        nulos = int((np.abs(f.coef_) < 1e-8).sum())
        print(f"{nombre} alfa={a:<6} EAM entr. {e:5.1f}"
              f"  prueba {p:5.1f}  coef. nulos {nulos}")

`h3_arbol.py`

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
from bicis import X, y, ent, pru

cols = ["hora", "laborable", "temp_c", "humedad", "viento_kmh",
        "tiempo", "mes", "año", "dia_semana", "festivo"]
for prof in [2, 4, 8, 12, 16, None]:
    a = DecisionTreeRegressor(max_depth=prof, random_state=0)
    a.fit(X.loc[ent, cols], y[ent])
    e = mean_absolute_error(y[ent], a.predict(X.loc[ent, cols]))
    p = mean_absolute_error(y[pru], a.predict(X.loc[pru, cols]))
    print(f"profundidad {str(prof):>4}: hojas {a.get_n_leaves():5d},"
          f" EAM entrenamiento {e:6.1f}, prueba {p:6.1f}")

`h4_conjuntos.py`

In [ ]:
import time
from sklearn.ensemble import (RandomForestRegressor,
                              HistGradientBoostingRegressor)
from sklearn.metrics import mean_absolute_error, r2_score
from bicis import X, y, ent, pru

cols = ["hora", "laborable", "temp_c", "humedad", "viento_kmh",
        "tiempo", "mes", "año", "dia_semana", "festivo"]
modelos = {
    "bosque aleatorio": RandomForestRegressor(
        300, min_samples_leaf=2, n_jobs=-1, random_state=0),
    "potenciación": HistGradientBoostingRegressor(
        max_iter=500, learning_rate=0.05, random_state=0),
    "potenciación Poisson": HistGradientBoostingRegressor(
        loss="poisson", max_iter=500, learning_rate=0.05,
        random_state=0)}
for nombre, m in modelos.items():
    t = time.perf_counter()
    p = m.fit(X.loc[ent, cols], y[ent]).predict(X.loc[pru, cols])
    print(f"{nombre:<21} EAM {mean_absolute_error(y[pru], p):5.1f}"
          f"  R² {r2_score(y[pru], p):.3f}"
          f"  ({time.perf_counter() - t:.0f} s)")

`h6_ajuste.py`

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
from bicis import X, y, ent, pru

cols = ["hora", "laborable", "temp_c", "humedad", "viento_kmh",
        "tiempo", "mes", "año", "dia_semana", "festivo"]
rejilla = {"learning_rate": [0.03, 0.1],
           "max_leaf_nodes": [15, 63],
           "l2_regularization": [0, 1]}
busca = GridSearchCV(
    HistGradientBoostingRegressor(loss="poisson", max_iter=300,
                                  random_state=0),
    rejilla, cv=TimeSeriesSplit(4),        # el futuro no se mira
    scoring="neg_mean_absolute_error").fit(X.loc[ent, cols], y[ent])
res = busca.cv_results_
for i in res["mean_test_score"].argsort()[::-1][:4]:
    q = res["params"][i]
    print(f"tasa {q['learning_rate']:<4}"
          f" hojas {q['max_leaf_nodes']:2d}"
          f" L2 {q['l2_regularization']}:"
          f" EAM val. {-res['mean_test_score'][i]:.1f}")
print("peor combinación: EAM val.",
      round(-res["mean_test_score"].min(), 1))
p = busca.predict(X.loc[pru, cols])
e = mean_absolute_error(y[pru], p)
print(f"mejor modelo en la prueba: EAM {e:.1f}")

`h8_curva.py`

In [ ]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from bicis import X, y, ent, pru

cols = ["hora", "laborable", "temp_c", "humedad", "viento_kmh",
        "tiempo", "mes", "año", "dia_semana", "festivo"]
idx = np.flatnonzero(ent)
rng = np.random.default_rng(0)
for frac in [0.05, 0.1, 0.25, 0.5, 1.0]:
    errores = []
    for rep in range(3):              # tres muestras distintas
        sel = rng.choice(idx, int(frac * len(idx)), replace=False)
        m = HistGradientBoostingRegressor(
            loss="poisson", max_iter=500, learning_rate=0.05,
            random_state=0).fit(X.loc[sel, cols], y[sel])
        p = m.predict(X.loc[pru, cols])
        errores.append(mean_absolute_error(y[pru], p))
    print(f"{frac:4.0%} ({len(sel):5d} horas): EAM prueba"
          f" {np.mean(errores):5.1f} ± {np.std(errores):3.1f}")

`h5_importancia.py`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import (permutation_importance,
                                PartialDependenceDisplay)
from bicis import X, y, ent, pru

cols = ["hora", "laborable", "temp_c", "humedad", "viento_kmh",
        "tiempo", "mes", "año", "dia_semana", "festivo"]
m = HistGradientBoostingRegressor(loss="poisson", max_iter=500,
                                  learning_rate=0.05, random_state=0)
m.fit(X.loc[ent, cols], y[ent])
imp = permutation_importance(m, X.loc[pru, cols], y[pru],
                             n_repeats=5, random_state=0,
                             scoring="neg_mean_absolute_error")
print("aumento del EAM al barajar cada variable:")
print(pd.Series(imp.importances_mean, cols).sort_values(
    ascending=False).round(1).to_string())
fig, ejes = plt.subplots(1, 2, figsize=(4.4, 1.9), sharey=True,
                         layout="constrained")
PartialDependenceDisplay.from_estimator(
    m, X.loc[ent, cols], ["temp_c", "humedad"], ax=ejes,
    method="brute", subsample=2000, random_state=0)
ejes[0].set_xlabel("temperatura (°C)")
ejes[1].set_xlabel("humedad (%)")
ejes[0].set_ylabel("alquileres por hora"); ejes[1].set_ylabel("")
plt.show()

**Prueba.** En `h4_conjuntos.py`, quita la variable `año`. ¿Cuánto empeora el EAM de prueba? ¿Por qué?